## Logistic Regression — Predicting Customer Churn

### Business Problem

A SaaS subscription business wants to identify customers at high risk of cancelling in the next 30 days. With a predicted churn probability for each account, the retention team can prioritize outreach and offer targeted incentives before revenue is lost.

### Why Logistic Regression?

This is a **binary classification** problem (churn vs. retain). Logistic regression outputs a calibrated **probability**, which is exactly what a prioritization queue needs — the team doesn't just need a yes/no, they need to rank accounts by risk level. It is also highly interpretable: each coefficient represents the log-odds contribution of a feature, allowing the team to understand *why* an account is flagged. It serves as an excellent baseline before investing in more complex models.

In [2]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [7]:
"""
Customer churn prediction using Logistic Regression.

Business goal: Score each customer account by churn probability
so the retention team can prioritize outreach.
"""


def generate_churn_data(n_samples: int = 1_000, seed: int = 42) -> pd.DataFrame:
    """Generate synthetic SaaS customer churn dataset.

    Args:
        n_samples: Number of customer records.
        seed: Random seed for reproducibility.

    Returns:
        DataFrame with behavioral features and churn label.
    """
    rng = np.random.default_rng(seed)

    days_since_login = rng.integers(1, 90, n_samples)
    support_tickets = rng.integers(0, 10, n_samples)
    monthly_spend = rng.uniform(50, 2_000, n_samples)
    contract_months_remaining = rng.integers(0, 24, n_samples)
    num_active_users = rng.integers(1, 50, n_samples)

    # Logit model to generate churn probabilities
    log_odds = (
        -2.0
        + (0.04 * days_since_login)
        + (0.3 * support_tickets)
        - (0.001 * monthly_spend)
        - (0.1 * contract_months_remaining)
        - (0.05 * num_active_users)
    )

    # Convert log-odds to probability using logistic function
    prob_churn = 1 / (1 + np.exp(-log_odds))
    churned = rng.binomial(1, prob_churn)

    return pd.DataFrame(
        {
            "days_since_login": days_since_login,
            "support_tickets_30d": support_tickets,
            "monthly_spend_usd": monthly_spend,
            "contract_months_remaining": contract_months_remaining,
            "num_active_users": num_active_users,
            "churned": churned,
        }
    )


"""Train and evaluate churn prediction model."""
df = generate_churn_data()
feature_cols = [c for c in df.columns if c != "churned"]

X = df[feature_cols]
y = df["churned"]

# Stratified split to maintain class balance in train/test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(class_weight="balanced", max_iter=1_000, random_state=42)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
y_proba = model.predict_proba(X_test_scaled)[:, 1]
auc = roc_auc_score(y_test, y_proba)

print("=== Logistic Regression: Churn Prediction ===")
print(f"ROC-AUC: {auc:.4f}\n")
print("""
Classification Report Summary:
- Precision: The proportion of predicted churned customers that were actually churned. A high precision means that when the model predicts a customer will churn, it is usually correct.
- Recall: The proportion of actual churned customers that were correctly identified by the model. A high recall means that the model is good at finding most of the churned customers.
- F1-score: The harmonic mean of precision and recall. It provides a single metric that balances both precision and recall, especially useful when there is an imbalance in the classes.
- Support: The number of actual occurrences of each class in the test set. This helps to understand how many samples were evaluated for each class.
""")
print(classification_report(y_test, y_pred, target_names=["Retained", "Churned"]))

# Create 10 new customer records for prediction
new_customers = generate_churn_data(n_samples=10, seed=999).drop(columns="churned")
new_customers_scaled = scaler.transform(new_customers)
new_churn_proba = model.predict_proba(new_customers_scaled)[:, 1]
new_customers["churn_probability"] = new_churn_proba
print("\nTop 5 New Customers at Risk:")
print(new_customers.nlargest(5, "churn_probability").to_string(index=False))

=== Logistic Regression: Churn Prediction ===
ROC-AUC: 0.9309


Classification Report Summary:
- Precision: The proportion of predicted churned customers that were actually churned. A high precision means that when the model predicts a customer will churn, it is usually correct.
- Recall: The proportion of actual churned customers that were correctly identified by the model. A high recall means that the model is good at finding most of the churned customers.
- F1-score: The harmonic mean of precision and recall. It provides a single metric that balances both precision and recall, especially useful when there is an imbalance in the classes.
- Support: The number of actual occurrences of each class in the test set. This helps to understand how many samples were evaluated for each class.

              precision    recall  f1-score   support

    Retained       0.97      0.83      0.89       164
     Churned       0.53      0.89      0.67        36

    accuracy                           